# Yorùbá OCR — Colab pipeline

**Runtime:** GPU → Runtime → Change runtime type → **T4 GPU**

| On Drive | Path |
|----------|------|
| Code + data | `My Drive/yor_ocr_research/` |
| Results | `yor_ocr_research/results/tables/` |
| HF cache | `yor_ocr_research/.hf_cache/` |

### What to run (default paper plan)

Edit **`RUN_PLAN`** in **Step 1** only. Everything else is Run All after setup.

| Section | What | Training? |
|---------|------|------------|
| **Setup** | Steps 0–5 — Drive, git pull, deps, data check | — |
| **A** | Out-of-the-box baselines | No — PP-OCR EN, **PaddleOCR-VL-1.6 ZS**, **GLM-OCR ZS** |
| **B** | **PaddleOCR-VL-1.6 SFT** | Yes — optional supervised fine-tune |
| **D** | Analysis + compile Table 1 | No |
| *Appendix* | HF dataset + model uploads | Optional toggles in Step 1 |

**Optional classical comparison:** PP-OCR recognition fine-tuning is available through `phase_04_train_paddleocr_recognition.sh`, but it is off by default and is not part of the active paper plan unless explicitly enabled.

**Run order after setup:** Section **A → B → D → Appendix**.

**Colab secret:** `HF_TOKEN` for model downloads / optional model upload.


## §0 Setup

### §0.1 Mount Google Drive

Run this cell **first** on Colab (before toggles or git pull).


In [ ]:
import os
import sys

try:
    from google.colab import drive  # type: ignore

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DRIVE_ROOT = "/content/drive/MyDrive" if IN_COLAB else None

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    os.chdir(DRIVE_ROOT)
    print("Mounted Google Drive.")
    print("cwd:", os.getcwd())
else:
    print("Not in Colab — skipping Drive mount.")


### §0.2 Run plan (edit this only)

Set a stable `RUN_ID` for one experiment. Re-running all cells with the same ID preserves partial inference and training checkpoints. Change the ID only when intentionally starting a fresh experiment.


In [ ]:
import os
import sys
from pathlib import Path

# Requires Step 0 on Colab (sets IN_COLAB, DRIVE_ROOT)
if "IN_COLAB" not in globals():
    try:
        import google.colab  # type: ignore  # noqa: F401
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False
if "DRIVE_ROOT" not in globals() or DRIVE_ROOT is None:
    DRIVE_ROOT = "/content/drive/MyDrive" if IN_COLAB else None

# ── Edit these values ────────────────────────────────────────────────────────
RUN_ID = "yoruba-ocr-final-v1"
RUN_RESET = True  # Safe on rerun: reset happens only once for this RUN_ID.

RUN_PLAN = {
    # Section A: OOTB zero-shot baselines (PP-OCR EN, PaddleOCR-VL-1.6, GLM-OCR)
    "A_baselines_ootb": True,
    # Section B: PaddleOCR-VL-1.6 LM fine-tuning
    "B_vl16_finetune": True,
    # Section D: Bootstrap CIs, stratified DER, compile CSVs
    "D_analysis_compile": True,
    # Appendix: HF dataset release
    "appendix_hf_dataset": False,
    # Appendix: HF model release
    "appendix_hf_models": False,
}
os.environ["OCR_RUN_ID"] = RUN_ID
os.environ["OCR_RUN_RESET"] = "1" if RUN_RESET else "0"
os.environ.setdefault("PADDLEOCRVL16_BATCH_SIZE", "2")
os.environ.setdefault("GLM_BATCH_SIZE", "2")
os.environ.setdefault("PADDLEOCRVL16_EVAL_BATCH_SIZE", "2")
os.environ.setdefault("SKIP_COMPLETED_EVAL", "1")

def _bootstrap_project_root():
    import os, sys
    from pathlib import Path
    candidates = []
    if os.environ.get("PROJECT_ROOT"):
        candidates.append(Path(os.environ["PROJECT_ROOT"]))
    if globals().get("REPO_DIR"):
        candidates.append(Path(globals()["REPO_DIR"]))
    candidates.extend([
        Path.cwd(),
        Path("/content/drive/MyDrive/research_ideas_and_how/yoruba_ocr_research"),
        Path("/content/drive/MyDrive/yor_ocr_research"),
        Path("/content/drive/MyDrive/yoruba_ocr_research"),
        Path("/kaggle/working/yoruba_ocr_research"),
    ])
    for cand in candidates:
        if (cand / "scripts" / "colab_stream.py").is_file():
            os.environ["PROJECT_ROOT"] = str(cand)
            if str(cand / "scripts") not in sys.path:
                sys.path.insert(0, str(cand / "scripts"))
            os.chdir(cand)
            return cand
    raise FileNotFoundError(
        "Could not locate project root with scripts/colab_stream.py. "
        "Rerun §0.3 Pull code from GitHub first."
    )

PROJECT_ROOT = _bootstrap_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)

try:
    from colab_run_plan import apply_run_plan, print_run_summary  # noqa: E402
    env = apply_run_plan(RUN_PLAN)
    print_run_summary(RUN_PLAN)
except ImportError:
    print("RUN_PLAN configured.")
    print("Note: colab_run_plan.py not found on disk yet (first-time setup).")
    print("The run plan will be applied automatically once the repository is cloned in Step 2.")


### §0.3 Pull code from GitHub

Updates **scripts/notebook only**. If you uploaded `data/processed/` before the first clone, Step 2 **stashes and restores** it — it does **not** wipe your Drive dataset.

First-time setup: upload `data/processed/` to `My Drive/yor_ocr_research/data/processed/`, then run Steps 0→2.


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

os.environ["PYTHON"] = sys.executable

# ── GitHub & Repository Configuration ───────────────────────────────────────
REPO_DIR_NAME = "yoruba_ocr_research"
GITHUB_REPO = "https://github.com/sam4rano/yoruba_ocr_research.git"
GITHUB_BRANCH = "main"
# Drive folder candidates — first existing repo/data folder wins.
# If Colab keeps picking the wrong folder, set REPO_DIR_OVERRIDE before this cell.
REPO_DIR_CANDIDATES = ["research_ideas_and_how/yoruba_ocr_research", "yor_ocr_research", REPO_DIR_NAME]
REPO_DIR_OVERRIDE = os.environ.get("REPO_DIR_OVERRIDE", "").strip()


def _has_uploaded_data(path: Path) -> bool:
    """True when Drive has a usable processed split (manual upload)."""
    return (path / "data" / "processed" / "labels" / "test.txt").is_file()


def _run(cmd: list[str], cwd: Path, *, check: bool = True) -> subprocess.CompletedProcess:
    """Run a command and optionally raise with captured output."""
    res = subprocess.run(cmd, cwd=str(cwd), capture_output=True, text=True)
    if check and res.returncode != 0:
        raise RuntimeError(
            f"Command failed ({res.returncode}): {' '.join(cmd)}\n"
            f"STDOUT:\n{res.stdout}\nSTDERR:\n{res.stderr}"
        )
    return res


def _init_git_repo(repo: Path) -> None:
    """Create a fresh git repo wrapper around existing data/results files."""
    repo.mkdir(parents=True, exist_ok=True)
    _run(["git", "init"], repo)
    remotes = _run(["git", "remote"], repo, check=False).stdout.split()
    if "origin" in remotes:
        _run(["git", "remote", "set-url", "origin", GITHUB_REPO], repo)
    else:
        _run(["git", "remote", "add", "origin", GITHUB_REPO], repo)


def _freshen_corrupt_git(repo: Path, reason: str) -> None:
    """Remove only git metadata when Drive corrupts shallow clone state."""
    print("\n" + "!" * 80)
    print("⚠️ Git metadata looks stale/corrupt; rebuilding .git only.")
    print("Reason:", reason.strip() or "unknown git failure")
    print("This preserves data/, results/, experiments/, and notebook outputs.")
    print("!" * 80 + "\n")
    git_dir = repo / ".git"
    if git_dir.exists():
        shutil.rmtree(git_dir)
    _init_git_repo(repo)


if IN_COLAB and DRIVE_ROOT:
    os.chdir(DRIVE_ROOT)

# Prefer explicit override, then folder with scripts/, then uploaded data/processed/.
if IN_COLAB and DRIVE_ROOT:
    if REPO_DIR_OVERRIDE:
        repo = Path(REPO_DIR_OVERRIDE)
        if not repo.is_absolute():
            repo = Path(DRIVE_ROOT) / repo
    else:
        candidates = [Path(DRIVE_ROOT) / name for name in REPO_DIR_CANDIDATES]
        repo = next((p for p in candidates if (p / "scripts").is_dir()), None)
        if repo is None:
            repo = next((p for p in candidates if _has_uploaded_data(p)), candidates[0])
    REPO_DIR = str(repo)
    os.environ["PROJECT_ROOT"] = REPO_DIR
    os.environ["HF_HOME"] = f"{REPO_DIR}/.hf_cache"
    os.environ["HF_HUB_CACHE"] = os.environ["HF_HOME"]
else:
    repo = Path.cwd()
    REPO_DIR = str(repo)

repo.parent.mkdir(parents=True, exist_ok=True)

# Initialize or update repository in-place to avoid Google Drive FUSE folder-exists errors
# and preserve any pre-uploaded data/ and results/ directories.
if not (repo / ".git").is_dir():
    print(f"Setting up repository in-place at {repo}...")
    _init_git_repo(repo)
else:
    print(f"Existing git repository detected at {repo}. Updating remote URL...")
    _run(["git", "remote", "set-url", "origin", GITHUB_REPO], repo)

print(f"Fetching origin/{GITHUB_BRANCH} (depth=1)...")
fetch_cmd = ["git", "fetch", "--depth", "1", "origin", GITHUB_BRANCH]
result = _run(fetch_cmd, repo, check=False)
if result.returncode != 0:
    err = result.stderr.strip()
    print(f"git fetch failed (exit {result.returncode}):\n{err}")
    recoverable = (
        "shallow file has changed" in err
        or "fatal: not a git repository" in err
        or "index file corrupt" in err
        or "bad object" in err
        or "unable to read" in err
    )
    if recoverable:
        _freshen_corrupt_git(repo, err)
        print(f"Retrying fetch origin/{GITHUB_BRANCH} after .git rebuild...")
        result = _run(fetch_cmd, repo, check=False)
    if result.returncode != 0:
        raise RuntimeError(
            f"git fetch still failed after recovery.\nSTDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}\n"
            "If this persists, set REPO_DIR_OVERRIDE to the exact Drive folder you want, "
            "or manually delete that folder's .git directory in Colab."
        )

# Check for uncommitted modifications to tracked files to avoid overwriting user edits.
status_res = _run(["git", "status", "--porcelain"], repo, check=False)
local_mods = [
    line[3:].strip()
    for line in status_res.stdout.splitlines()
    if line.startswith((" M", "M ", " D", "D "))
]
if local_mods:
    print("\n" + "!" * 80)
    print("⚠️ WARNING: Local modifications detected in tracked scripts/files:")
    for mod in local_mods:
        print(f"  - {mod}")
    print("Automatically stashing your changes before resetting/updating from GitHub...")
    _run(["git", "stash", "push", "-m", "Auto-stash before notebook update"], repo, check=False)
    print("Your edits have been saved to the git stash. You can pop them back via:")
    print(f"  git -C {repo} stash pop")
    print("!" * 80 + "\n")

print("Checking out and resetting to origin/main...")
_run(["git", "checkout", "-f", "-B", GITHUB_BRANCH, f"origin/{GITHUB_BRANCH}"], repo)
_run(["git", "reset", "--hard", f"origin/{GITHUB_BRANCH}"], repo)

# A matching Git ref is insufficient: the opened notebook may be newer than GitHub.
# Validate the current descriptive layout before any expensive install or GPU phase.
required_layout = [
    "notebooks/colab_ocr.ipynb",
    "scripts/initialize_notebook_run.py",
    "scripts/smoke_test_runtime.py",
    "scripts/refresh_dataset_report.py",
    "scripts/audit_data_quality.py",
    "scripts/check_completed_metric.py",
    "scripts/vl_eval_runtime.py",
]
missing_layout = [name for name in required_layout if not (repo / name).is_file()]
if missing_layout:
    legacy_layout = (repo / "scripts/24_colab_smoke_test.py").is_file()
    layout_kind = "legacy numbered-script layout" if legacy_layout else "incomplete layout"
    current_head = _run(["git", "rev-parse", "--short", "HEAD"], repo, check=False).stdout.strip()
    raise RuntimeError(
        f"Notebook/repository revision mismatch: origin/{GITHUB_BRANCH} at {current_head or 'unknown'} "
        f"has a {layout_kind} and is missing: {', '.join(missing_layout)}. "
        "Do not continue with later cells. The current OCR changes must first be committed "
        "and pushed together with notebooks/colab_ocr.ipynb; then rerun this pull cell."
    )
print("Repository code and notebook layout are aligned. data/ and results/ are untouched.")

os.chdir(repo)
DATA_DIR = repo / "data"
head = _run(["git", "rev-parse", "--short", "HEAD"], repo).stdout.strip()
remote_head = _run(["git", "rev-parse", "--short", f"origin/{GITHUB_BRANCH}"], repo).stdout.strip()
print("cwd:", os.getcwd())
print("git HEAD:", head, "| origin/main:", remote_head)
print("data/processed:", (DATA_DIR / "processed").is_dir())
print("data/raw:", (DATA_DIR / "raw").is_dir())

def _bootstrap_project_root():
    import os, sys
    from pathlib import Path
    candidates = []
    if os.environ.get("PROJECT_ROOT"):
        candidates.append(Path(os.environ["PROJECT_ROOT"]))
    if globals().get("REPO_DIR"):
        candidates.append(Path(globals()["REPO_DIR"]))
    candidates.extend([
        Path.cwd(),
        Path("/content/drive/MyDrive/research_ideas_and_how/yoruba_ocr_research"),
        Path("/content/drive/MyDrive/yor_ocr_research"),
        Path("/content/drive/MyDrive/yoruba_ocr_research"),
        Path("/kaggle/working/yoruba_ocr_research"),
    ])
    for cand in candidates:
        if (cand / "scripts" / "colab_stream.py").is_file():
            os.environ["PROJECT_ROOT"] = str(cand)
            if str(cand / "scripts") not in sys.path:
                sys.path.insert(0, str(cand / "scripts"))
            os.chdir(cand)
            return cand
    raise FileNotFoundError(
        "Could not locate project root with scripts/colab_stream.py. "
        "Rerun §0.3 Pull code from GitHub first."
    )

PROJECT_ROOT = _bootstrap_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)
from colab_stream import run_cmd, run_phase
from colab_run_plan import apply_run_plan, print_run_summary

apply_run_plan(RUN_PLAN)
print_run_summary(RUN_PLAN)
print("run_cmd / run_phase ready — long steps stream logs live.")

# Auto-respect existing processed data unless user forced consolidate.
test_labels = DATA_DIR / "processed" / "labels" / "test.txt"
if os.environ.get("USE_EXISTING_PROCESSED_DATA", "1") == "1" and test_labels.is_file():
    os.environ["SKIP_CONSOLIDATE"] = "1"
    os.environ["RUN_RESPLIT"] = "0"
    with test_labels.open(encoding="utf-8") as fh:
        n = sum(1 for _ in fh)
    print(f"Using existing data/processed (test.txt: {n} lines). SKIP_CONSOLIDATE=1")


### §0.4 Install dependencies


In [ ]:
import subprocess, sys, os
from pathlib import Path

if not Path("requirements.txt").is_file():
    raise RuntimeError("Run §0.3 (git pull) first.")

PY = sys.executable

def pip(*args):
    subprocess.check_call([PY, "-m", "pip", "install", "-q", *args])

if IN_COLAB:
    pip("paddlepaddle-gpu", "-f", "https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html")

processed_reqs = []
for ln in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    ln = ln.strip()
    if not ln or ln.startswith("#"):
        continue
    # Skip paddlepaddle (installed separately)
    if ln.startswith("paddlepaddle"):
        continue
    # Fix editdistance pin (to avoid Python 3.11/3.12 wheel search failures)
    if "editdistance" in ln:
        ln = "editdistance>=0.6.2"
    # Comment out opencv-python pin (conflicts with paddleocr <=4.6.0.66)
    if "opencv-python" in ln:
        continue
    # Fix pandas pin (to avoid Python 3.12 trying to build pandas 2.0.x from source)
    if "pandas" in ln:
        ln = "pandas>=2.0.0"
    # Fix transformers pin
    # Fix accelerate pin (conflicts with transformers>=5)
    if "accelerate" in ln:
        ln = "accelerate>=1.1.0"
    if "transformers" in ln:
        ln = "transformers>=5.0.0"
    processed_reqs.append(ln)

Path("/tmp/reqs_no_paddle.txt").write_text("\n".join(processed_reqs) + "\n", encoding="utf-8")
pip("-r", "/tmp/reqs_no_paddle.txt")

# Install/upgrade the HF stack last. PaddleOCR-VL-1.6 and GLM-OCR need modern
# transformers/accelerate; restart the runtime if imports still see an older version.
pip("transformers>=5.0.0", "accelerate>=1.1.0", "datasets", "safetensors", "huggingface_hub>=1.5.0", "einops", "torchvision")
pip("bitsandbytes")   # optional 4-bit quant for PaddleOCR-VL-1.6 / GLM-OCR on T4
# Force reinstall numpy 1.x and pandas to resolve C-API binary incompatibility (Expected 96, got 88)
pip("numpy>=1.24.0,<2.0.0", "pandas>=2.0.0", "--force-reinstall")

if IN_COLAB:
    res = subprocess.run("apt-get update -qq && apt-get install -qq -y libgl1", shell=True, check=False)
    if res.returncode != 0:
        print("WARNING: apt-get install libgl1 failed. OpenCV might raise shared-library errors later.")
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            os.environ["HF_TOKEN"] = tok
            os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", tok)
            from huggingface_hub import login
            login(token=tok)
            print("HF_TOKEN loaded from Colab secret")
    except Exception as exc:
        print("HF_TOKEN not set:", exc)

import paddle, torch, transformers, accelerate
print("Paddle", paddle.__version__, "CUDA", paddle.device.is_compiled_with_cuda())
print("Torch", torch.__version__, "CUDA", torch.cuda.is_available())
print("Transformers", transformers.__version__, "Accelerate", accelerate.__version__)
if int(transformers.__version__.split(".")[0]) < 5:
    raise RuntimeError("transformers>=5 is required. Restart the Colab runtime, then rerun setup cells.")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


### §0.5 Clone PaddleOCR + extras


In [ ]:
import shutil, subprocess, sys
from pathlib import Path

pdir = Path("PaddleOCR")
if pdir.is_dir() and not (pdir / "requirements.txt").is_file():
    shutil.rmtree(pdir)
if not pdir.is_dir():
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", "main",
                           "https://github.com/PaddlePaddle/PaddleOCR.git", "PaddleOCR"])
if (pdir / "requirements.txt").is_file():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "PaddleOCR/requirements.txt"])

# PaddleOCR requirements can pull older transitive deps. Re-assert the VLM stack.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=5.0.0", "accelerate>=1.1.0", "huggingface_hub>=1.5.0",
    "einops", "torchvision", "bitsandbytes"
])
print("PaddleOCR OK; VLM dependency stack re-checked")


### §0.6 Pre-flight smoke test (no GPU training)

Run **after Step 3** (deps installed). Uses `--quick`: data layout, config, VL JSONL export — **does not** require eval JSONL yet. For the full post-evaluation check (error analyses and HF card), pass `--full` after Step 11.


In [ ]:
import os, subprocess, sys
from pathlib import Path

def _bootstrap_project_root():
    import os, sys
    from pathlib import Path
    candidates = []
    if os.environ.get("PROJECT_ROOT"):
        candidates.append(Path(os.environ["PROJECT_ROOT"]))
    if globals().get("REPO_DIR"):
        candidates.append(Path(globals()["REPO_DIR"]))
    candidates.extend([
        Path.cwd(),
        Path("/content/drive/MyDrive/research_ideas_and_how/yoruba_ocr_research"),
        Path("/content/drive/MyDrive/yor_ocr_research"),
        Path("/content/drive/MyDrive/yoruba_ocr_research"),
        Path("/kaggle/working/yoruba_ocr_research"),
    ])
    for cand in candidates:
        if (cand / "scripts" / "colab_stream.py").is_file():
            os.environ["PROJECT_ROOT"] = str(cand)
            if str(cand / "scripts") not in sys.path:
                sys.path.insert(0, str(cand / "scripts"))
            os.chdir(cand)
            return cand
    raise FileNotFoundError(
        "Could not locate project root with scripts/colab_stream.py. "
        "Rerun §0.3 Pull code from GitHub first."
    )

PROJECT_ROOT = _bootstrap_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)
from colab_stream import run_cmd

repo = Path(REPO_DIR)
os.environ["PYTHON"] = sys.executable
os.environ["PROJECT_ROOT"] = str(repo)
run_cmd(
    [
        sys.executable,
        "scripts/smoke_test_runtime.py",
        "--quick",
        "--skip-config",
        "--network",
    ],
    cwd=repo,
)
print("Smoke + network checks passed — safe to continue to baselines .")


## §1 Data & Config (Phases 01–03)

With `USE_EXISTING_PROCESSED_DATA=1` (default), **Phase 01 is skipped** and your Drive `data/processed/` is used as-is.

Set `SKIP_CONSOLIDATE=0` and `RUN_RESPLIT=1` in Step 0 only when rebuilding from `data/raw/`.


In [ ]:
import os, subprocess, sys, json, shutil
from pathlib import Path

def _bootstrap_project_root():
    import os, sys
    from pathlib import Path
    candidates = []
    if os.environ.get("PROJECT_ROOT"):
        candidates.append(Path(os.environ["PROJECT_ROOT"]))
    if globals().get("REPO_DIR"):
        candidates.append(Path(globals()["REPO_DIR"]))
    candidates.extend([
        Path.cwd(),
        Path("/content/drive/MyDrive/research_ideas_and_how/yoruba_ocr_research"),
        Path("/content/drive/MyDrive/yor_ocr_research"),
        Path("/content/drive/MyDrive/yoruba_ocr_research"),
        Path("/kaggle/working/yoruba_ocr_research"),
    ])
    for cand in candidates:
        if (cand / "scripts" / "colab_stream.py").is_file():
            os.environ["PROJECT_ROOT"] = str(cand)
            if str(cand / "scripts") not in sys.path:
                sys.path.insert(0, str(cand / "scripts"))
            os.chdir(cand)
            return cand
    raise FileNotFoundError(
        "Could not locate project root with scripts/colab_stream.py. "
        "Rerun §0.3 Pull code from GitHub first."
    )

PROJECT_ROOT = _bootstrap_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)
from colab_stream import run_cmd, run_phase

PY = sys.executable
Path("results/tables").mkdir(parents=True, exist_ok=True)
run_cmd([
    PY, "scripts/initialize_notebook_run.py",
    "--run-id", os.environ["OCR_RUN_ID"],
    *(["--reset"] if os.environ.get("OCR_RUN_RESET") == "1" else []),
])

if os.environ.get("SKIP_CONSOLIDATE", "1") == "1":
    print("SKIP_CONSOLIDATE=1 — using existing data/processed on Drive (not creating empty data/raw)")
else:
    Path("data/raw").mkdir(parents=True, exist_ok=True)
    if os.environ.get("RESET_PROCESSED", "0") == "1" and Path("data/processed").is_dir():
        shutil.rmtree("data/processed")
    cmd = [PY, "scripts/consolidate_data.py",
           "--raw-dir", "data/raw", "--output-dir", "data/processed",
           "--log-file", "results/tables/consolidation_report.json"]
    if os.environ.get("RUN_RESPLIT", "0") == "1":
        cmd += ["--resplit", "--seed", "42", "--train-ratio", "0.8", "--val-ratio", "0.1", "--test-ratio", "0.1"]
    run_cmd(cmd)

checks = {
    "data/processed/labels/train.txt": "file",
    "data/processed/labels/val.txt": "file",
    "data/processed/labels/test.txt": "file",
    "data/processed/dictionary/yoruba_char_dict.txt": "file",
    "data/processed/images/test": "dir",
}
for path, kind in checks.items():
    p = Path(path)
    ok = p.is_file() if kind == "file" else p.is_dir()
    extra = ""
    if ok and kind == "file":
        with p.open(encoding="utf-8") as fh:
            line_count = sum(1 for _ in fh)
        extra = f" ({line_count} lines)"
    elif ok:
        valid_exts = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}
        img_files = [f for f in p.glob("*") if f.suffix.lower() in valid_exts]
        extra = f" ({len(img_files)} images)"
    print("OK" if ok else "MISSING", path, extra)
    if not ok:
        raise RuntimeError("Upload data to Drive under data/raw/ or data/processed/")

rep = Path("results/tables/consolidation_report.json")
if rep.is_file():
    try:
        r = json.loads(rep.read_text(encoding="utf-8"))
        sc = r.get("split_counts", {})
        print(f"consolidation: n={r.get('unique_images_total')} train={sc.get('train')} val={sc.get('val')} test={sc.get('test')}")
    except json.JSONDecodeError as exc:
        print(f"WARNING: consolidation_report.json is malformed or partially written: {exc}")

run_cmd([PY, "scripts/refresh_dataset_report.py"])
run_cmd([PY, "scripts/audit_data_quality.py", "--data-dir", "data/processed", "--out-json", "results/tables/data_quality.json"])
run_phase("phase_02_analyze.sh")
env = os.environ.copy()
env["CONFIG_FORCE_GPU"] = "1"
run_phase("phase_03_config.sh", env=env)
print("Phases 01-03 done")


In [ ]:

def _bootstrap_project_root():
    import os, sys
    from pathlib import Path
    candidates = []
    if os.environ.get("PROJECT_ROOT"):
        candidates.append(Path(os.environ["PROJECT_ROOT"]))
    if globals().get("REPO_DIR"):
        candidates.append(Path(globals()["REPO_DIR"]))
    candidates.extend([
        Path.cwd(),
        Path("/content/drive/MyDrive/research_ideas_and_how/yoruba_ocr_research"),
        Path("/content/drive/MyDrive/yor_ocr_research"),
        Path("/content/drive/MyDrive/yoruba_ocr_research"),
        Path("/kaggle/working/yoruba_ocr_research"),
    ])
    for cand in candidates:
        if (cand / "scripts" / "colab_stream.py").is_file():
            os.environ["PROJECT_ROOT"] = str(cand)
            if str(cand / "scripts") not in sys.path:
                sys.path.insert(0, str(cand / "scripts"))
            os.chdir(cand)
            return cand
    raise FileNotFoundError(
        "Could not locate project root with scripts/colab_stream.py. "
        "Rerun §0.3 Pull code from GitHub first."
    )

PROJECT_ROOT = _bootstrap_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)

# Re-run data quality audit after Phase 02 to ensure image dimensions are captured
import sys
from colab_stream import run_cmd  # noqa: F811
run_cmd([sys.executable, "scripts/audit_data_quality.py", "--data-dir", "data/processed", "--out-json", "results/tables/data_quality.json"])


In [ ]:
# Freeze split labels to data/splits/ for reproducibility if not already done
import shutil
from pathlib import Path
splits_dir = Path("data/splits")
splits_dir.mkdir(parents=True, exist_ok=True)
for split in ("train", "val", "test"):
    src = Path("data/processed/labels") / f"{split}.txt"
    dest = splits_dir / f"{split}.txt"
    if src.is_file():
        shutil.copy2(src, dest)
        print(f"Frozen {split}.txt → {dest}")


## §2 Baselines (Section A — OOTB zero-shot evals)

PP-OCR English pretrained · **PaddleOCR-VL-1.6 zero-shot** · **GLM-OCR zero-shot**

Skipped when `A_baselines_ootb=False`.


In [ ]:
import gc, os, subprocess, sys
from pathlib import Path

def _bootstrap_project_root():
    import os, sys
    from pathlib import Path
    candidates = []
    if os.environ.get("PROJECT_ROOT"):
        candidates.append(Path(os.environ["PROJECT_ROOT"]))
    if globals().get("REPO_DIR"):
        candidates.append(Path(globals()["REPO_DIR"]))
    candidates.extend([
        Path.cwd(),
        Path("/content/drive/MyDrive/research_ideas_and_how/yoruba_ocr_research"),
        Path("/content/drive/MyDrive/yor_ocr_research"),
        Path("/content/drive/MyDrive/yoruba_ocr_research"),
        Path("/kaggle/working/yoruba_ocr_research"),
    ])
    for cand in candidates:
        if (cand / "scripts" / "colab_stream.py").is_file():
            os.environ["PROJECT_ROOT"] = str(cand)
            if str(cand / "scripts") not in sys.path:
                sys.path.insert(0, str(cand / "scripts"))
            os.chdir(cand)
            return cand
    raise FileNotFoundError(
        "Could not locate project root with scripts/colab_stream.py. "
        "Rerun §0.3 Pull code from GitHub first."
    )

PROJECT_ROOT = _bootstrap_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)
from colab_stream import run_cmd, run_phase

def _free_gpu_memory(label=""):
    import gc
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            free, total = torch.cuda.mem_get_info()
            print(f"GPU cleanup {label}: {free / (1024**3):.2f} GB free / {total / (1024**3):.2f} GB total")
    except Exception as exc:
        print(f"GPU cleanup {label}: skipped ({exc})")


try:
    from IPython.display import display
except ImportError:
    display = print

if os.environ.get("RUN_PLAN_A", "1") != "1":
    print("Section A skipped (A_baselines_ootb=False)")
else:
    env = os.environ.copy()
    env.setdefault("SKIP_PADDLE_FINETUNE", "1")
    env["EVAL_USE_GPU"] = "1"

    # 1. Base PaddleOCR -- PP-OCR English pretrained
    run_phase("phase_05_eval_paddleocr_recognition.sh", env=env, label="PaddleOCR EN pretrained")
    _free_gpu_memory("after PP-OCR baseline")

    # Flush Paddle CUDA context before loading PyTorch VLM models
    # to prevent torch.cuda.is_available() from returning False.
    gc.collect()
    try:
        import torch as _t
        if _t.cuda.is_available():
            _t.cuda.empty_cache()
    except Exception:
        pass

    # 2. PaddleOCR-VL-1.6 zero-shot
    # PADDLEOCRVL16_QUANTIZE_4BIT=0 (default): use hardware-native dtype (bf16 on L4/A100, fp16 on T4).
    # PADDLEOCRVL16_QUANTIZE_4BIT=1:           use 4-bit only when VRAM < 6 GB.
    if env.get("SKIP_PADDLEOCRVL16_ZERO_SHOT", "0") == "0":
        env["PADDLEOCRVL16_QUANTIZE_4BIT"] = env.get("PADDLEOCRVL16_QUANTIZE_4BIT", "0")
        # Ensure no stale sample-cap from previous debug runs leaks into full eval.
        env.pop("PADDLEOCRVL16_MAX_SAMPLES", None)
        run_phase("phase_15_eval_paddleocrvl16_zero_shot.sh", env=env, label="PaddleOCR-VL-1.6 zero-shot")
        _free_gpu_memory("after PaddleOCR-VL zero-shot")
    else:
        print("SKIP_PADDLEOCRVL16_ZERO_SHOT=1")

    # Flush CUDA cache between large VLM loads
    gc.collect()
    try:
        if _t.cuda.is_available():
            _t.cuda.empty_cache()
    except Exception:
        pass

    # 3. GLM-OCR zero-shot
    # GLM_QUANTIZE_4BIT=0 (default): use hardware-native dtype (bf16 on L4/A100, fp16 on T4).
    if env.get("SKIP_GLM_ZERO_SHOT", "0") == "0":
        env["GLM_QUANTIZE_4BIT"] = env.get("GLM_QUANTIZE_4BIT", "0")
        # Ensure no stale sample-cap from previous debug runs leaks into full eval.
        env.pop("GLM_MAX_SAMPLES", None)
        run_phase("phase_18_eval_glm_ocr_zero_shot.sh", env=env, label="GLM-OCR zero-shot")
        _free_gpu_memory("after GLM-OCR zero-shot")
    else:
        print("SKIP_GLM_ZERO_SHOT=1")

    from pathlib import Path
    mp = Path("results/tables/metrics.csv")
    if mp.is_file():
        try:
            import pandas as pd
            df = pd.read_csv(mp)
            display(df.tail(10))
        except Exception as exc:
            print(f"metrics.csv was updated, but pandas display failed: {exc}")
            print("This usually means Colab needs Runtime → Restart session after dependency installs.")
            print("Last metrics rows:")
            lines = mp.read_text(encoding="utf-8").splitlines()
            for line in lines[-10:]:
                print(line)


## §3 Fine-tuned Models (Section B — VL-1.6 SFT)

### §3.1 PaddleOCR-VL-1.6 SFT

**B1 — PaddleOCR-VL-1.6:** verify the Section A zero-shot row → export JSONL → **SFT train** → SFT eval.

Skipped when `B_vl16_finetune=False`.


In [ ]:
import os, sys
from pathlib import Path


def _bootstrap_project_root():
    import os, sys
    from pathlib import Path
    candidates = []
    if os.environ.get("PROJECT_ROOT"):
        candidates.append(Path(os.environ["PROJECT_ROOT"]))
    if globals().get("REPO_DIR"):
        candidates.append(Path(globals()["REPO_DIR"]))
    candidates.extend([
        Path.cwd(),
        Path("/content/drive/MyDrive/research_ideas_and_how/yoruba_ocr_research"),
        Path("/content/drive/MyDrive/yor_ocr_research"),
        Path("/content/drive/MyDrive/yoruba_ocr_research"),
        Path("/kaggle/working/yoruba_ocr_research"),
    ])
    for cand in candidates:
        if (cand / "scripts" / "colab_stream.py").is_file():
            os.environ["PROJECT_ROOT"] = str(cand)
            if str(cand / "scripts") not in sys.path:
                sys.path.insert(0, str(cand / "scripts"))
            os.chdir(cand)
            return cand
    raise FileNotFoundError(
        "Could not locate project root with scripts/colab_stream.py. "
        "Rerun §0.3 Pull code from GitHub first."
    )

PROJECT_ROOT = _bootstrap_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)
from colab_stream import run_phase

if os.environ.get("RUN_PLAN_B", "1") != "1":
    print("Phase 14 export skipped (B_vl16_finetune=False)")
else:
    run_phase("phase_14_export_paddleocrvl16_sft.sh", label="VL-1.6 SFT export")
    for split in ("train", "val", "test"):
        f = Path("data/paddleocrvl16_sft") / f"{split}.jsonl"
        if f.is_file():
            with f.open(encoding="utf-8") as fh:
                n = sum(1 for _ in fh)
        else:
            n = 0
        print(f"{split}.jsonl: {n}")

mp = Path("results/tables/metrics.csv")
if mp.is_file():
    try:
        import pandas as pd
        df = pd.read_csv(mp)
        col = "model_name" if "model_name" in df.columns else "model"
        zs = df[df[col] == "paddleocrvl16_zero_shot"]
        print("zero-shot row:", "OK" if not zs.empty else "MISSING")
    except Exception as exc:
        text = mp.read_text(encoding="utf-8")
        print(f"Could not import/read pandas ({exc}); checking metrics.csv as text.")
        print("zero-shot row:", "OK" if "paddleocrvl16_zero_shot" in text else "MISSING")


In [ ]:
import os, sys
from pathlib import Path

import pandas as pd

def _bootstrap_project_root():
    import os, sys
    from pathlib import Path
    candidates = []
    if os.environ.get("PROJECT_ROOT"):
        candidates.append(Path(os.environ["PROJECT_ROOT"]))
    if globals().get("REPO_DIR"):
        candidates.append(Path(globals()["REPO_DIR"]))
    candidates.extend([
        Path.cwd(),
        Path("/content/drive/MyDrive/research_ideas_and_how/yoruba_ocr_research"),
        Path("/content/drive/MyDrive/yor_ocr_research"),
        Path("/content/drive/MyDrive/yoruba_ocr_research"),
        Path("/kaggle/working/yoruba_ocr_research"),
    ])
    for cand in candidates:
        if (cand / "scripts" / "colab_stream.py").is_file():
            os.environ["PROJECT_ROOT"] = str(cand)
            if str(cand / "scripts") not in sys.path:
                sys.path.insert(0, str(cand / "scripts"))
            os.chdir(cand)
            return cand
    raise FileNotFoundError(
        "Could not locate project root with scripts/colab_stream.py. "
        "Rerun §0.3 Pull code from GitHub first."
    )

PROJECT_ROOT = _bootstrap_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)
from colab_stream import run_cmd, run_phase

def _free_gpu_memory(label=""):
    import gc
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            free, total = torch.cuda.mem_get_info()
            print(f"GPU cleanup {label}: {free / (1024**3):.2f} GB free / {total / (1024**3):.2f} GB total")
    except Exception as exc:
        print(f"GPU cleanup {label}: skipped ({exc})")


try:
    from IPython.display import display
except ImportError:
    display = print

if os.environ.get("RUN_PLAN_B", "1") != "1":
    print("Section B PaddleOCR-VL-1.6 SFT skipped (B_vl16_finetune=False)")
else:
    # VRAM pre-flight check — SFT training is the heaviest GPU phase
    try:
        import torch as _torch
        if _torch.cuda.is_available():
            _free, _total = _torch.cuda.mem_get_info()
            _free_gb  = _free  / (1024 ** 3)
            _total_gb = _total / (1024 ** 3)
            print(f"VRAM: {_free_gb:.1f} GB free / {_total_gb:.1f} GB total")
            if _free_gb < 8.0:
                print(
                    f"WARNING: Only {_free_gb:.1f} GB VRAM free. "
                    "VL-1.6 4-bit SFT typically needs ≥8 GB. "
                    "Consider restarting the runtime to free memory before training."
                )
        else:
            print("WARNING: No CUDA GPU detected — training will run on CPU (very slow).")
    except ImportError:
        print("torch not yet installed — VRAM check skipped.")
    env = {**os.environ, "EVAL_USE_GPU": "1", "PYTHONUNBUFFERED": "1"}
    ckpt_dir = Path("experiments/paddleocrvl16_sft")
    if ckpt_dir.is_dir() and (ckpt_dir / "training_state.json").is_file():
        print(f"Found existing fine-tuning checkpoint state at {ckpt_dir}.")
        print("Resuming training from the last saved state (PADDLEOCRVL16_SFT_RESUME=1).")
        env["PADDLEOCRVL16_SFT_RESUME"] = "1"

    if os.environ.get("SKIP_PADDLEOCRVL16_SFT_TRAIN", "0") == "0":
        # Remove sample-cap from the env snapshot passed to training.
        # Verified: phase_16_train_paddleocrvl16_sft.sh reads PADDLEOCRVL16_SFT_MAX_SAMPLES;
        # phase_17 uses PADDLEOCRVL16_EVAL_MAX_SAMPLES — no cross-phase leakage.
        env.pop("PADDLEOCRVL16_SFT_MAX_SAMPLES", None)

        # Colab/Kaggle T4 safety defaults for PaddleOCR-VL-1.6 SFT.
        # Use lm_head first: it is cheaper, less overfit-prone on 2.3k lines,
        # and gives a fast signal whether Yorùbá supervision is helping.
        # Override to non_vision only on larger GPUs after lm_head proves useful.
        env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
        env.setdefault("PADDLEOCRVL16_SFT_TRAIN_SCOPE", "lm_head")
        env.setdefault("PADDLEOCRVL16_SFT_LR", "5e-5")
        env.setdefault("PADDLEOCRVL16_SFT_EPOCHS", "3")
        env.setdefault("PADDLEOCRVL16_SFT_MAX_PIXELS", str(384 * 28 * 28))
        env.setdefault("PADDLEOCRVL16_SFT_GRAD_ACCUM", "32")
        env.setdefault("PADDLEOCRVL16_SFT_VAL_SAMPLES", "64")
        env.setdefault("PADDLEOCRVL16_SFT_EVAL_MAX_NEW_TOKENS", "192")
        env.setdefault("PADDLEOCRVL16_SFT_EMPTY_CACHE_STEPS", "10")
        if (ckpt_dir / "training_state.json").is_file():
            env.setdefault("PADDLEOCRVL16_SFT_RESUME", "1")
        print("SFT memory settings:")
        print("  PYTORCH_CUDA_ALLOC_CONF=", env.get("PYTORCH_CUDA_ALLOC_CONF"))
        print("  PADDLEOCRVL16_SFT_TRAIN_SCOPE=", env.get("PADDLEOCRVL16_SFT_TRAIN_SCOPE"))
        print("  PADDLEOCRVL16_SFT_LR=", env.get("PADDLEOCRVL16_SFT_LR"))
        print("  PADDLEOCRVL16_SFT_EPOCHS=", env.get("PADDLEOCRVL16_SFT_EPOCHS"))
        print("  PADDLEOCRVL16_SFT_MAX_PIXELS=", env.get("PADDLEOCRVL16_SFT_MAX_PIXELS"))
        print("  PADDLEOCRVL16_SFT_GRAD_ACCUM=", env.get("PADDLEOCRVL16_SFT_GRAD_ACCUM"))
        print("  PADDLEOCRVL16_SFT_VAL_SAMPLES=", env.get("PADDLEOCRVL16_SFT_VAL_SAMPLES"))
        print("  PADDLEOCRVL16_SFT_EVAL_MAX_NEW_TOKENS=", env.get("PADDLEOCRVL16_SFT_EVAL_MAX_NEW_TOKENS"))
        print("  PADDLEOCRVL16_SFT_EMPTY_CACHE_STEPS=", env.get("PADDLEOCRVL16_SFT_EMPTY_CACHE_STEPS"))
        print("  PADDLEOCRVL16_SFT_RESUME=", env.get("PADDLEOCRVL16_SFT_RESUME", "0"))

        run_phase("phase_16_train_paddleocrvl16_sft.sh", env=env, label="Phase 16 VL-1.6 Fine-Tuning")
        log_path = Path("experiments/paddleocrvl16_sft/training_log.jsonl")
        if log_path.is_file():
            print("\nLatest SFT validation/training log rows:")
            rows = log_path.read_text(encoding="utf-8").strip().splitlines()[-5:]
            for row in rows:
                print(row)
        _free_gpu_memory("after VL SFT training")
    else:
        print("SKIP_PADDLEOCRVL16_SFT_TRAIN=1")

    # Evaluate the fine-tuned model
    if (Path("experiments/paddleocrvl16_sft/best/config.json").is_file() or
            Path("experiments/paddleocrvl16_sft/config.json").is_file()):
        run_phase("phase_17_eval_paddleocrvl16_sft.sh", env=env, label="VL-1.6 fine-tuned eval")
        _free_gpu_memory("after VL SFT eval")
    else:
        print("SFT evaluation skipped: no completed checkpoint found.")

    mp = Path("results/tables/metrics.csv")
    if mp.is_file():
        df = pd.read_csv(mp)
        col = "model_name" if "model_name" in df.columns else "model"
        vl = df[df[col].str.startswith("paddleocrvl16", na=False)]
        display(vl[[c for c in (col, "n", "cer", "wer", "der") if c in vl.columns]].to_string(index=False))
        base = vl[vl[col] == "paddleocrvl16_zero_shot"]
        tuned = vl[vl[col] == "paddleocrvl16_sft"]
        if not base.empty and not tuned.empty:
            base_cer = float(base.iloc[-1]["cer"])
            tuned_cer = float(tuned.iloc[-1]["cer"])
            rel = (base_cer - tuned_cer) / base_cer * 100 if base_cer else 0.0
            print(f"SFT vs zero-shot CER: {base_cer:.4f} → {tuned_cer:.4f} ({rel:+.2f}% relative reduction)")
            if tuned_cer >= base_cer:
                print("WARNING: fine-tuning did not improve test CER; do not claim an improvement. Review the epoch validation log and training scope.")
        else:
            print("Comparison pending: both zero-shot and SFT metric rows are required.")


## §4 Optional Classical PaddleOCR Fine-Tune

This is an optional supervised PaddleOCR recognition comparison, not a default paper section. It uses `scripts/train_paddleocr_recognition.py`, which delegates to `PaddleOCR/tools/train.py` and writes `results/tables/train_run.json`.

Default: skipped with `SKIP_PADDLE_TRAIN=1`. Set `SKIP_PADDLE_TRAIN=0` only when you intentionally want the long classical training run.


In [ ]:
import os, sys
from pathlib import Path

def _bootstrap_project_root():
    import os, sys
    from pathlib import Path
    candidates = []
    if os.environ.get("PROJECT_ROOT"):
        candidates.append(Path(os.environ["PROJECT_ROOT"]))
    if globals().get("REPO_DIR"):
        candidates.append(Path(globals()["REPO_DIR"]))
    candidates.extend([
        Path.cwd(),
        Path("/content/drive/MyDrive/research_ideas_and_how/yoruba_ocr_research"),
        Path("/content/drive/MyDrive/yor_ocr_research"),
        Path("/content/drive/MyDrive/yoruba_ocr_research"),
        Path("/kaggle/working/yoruba_ocr_research"),
    ])
    for cand in candidates:
        if (cand / "scripts" / "colab_stream.py").is_file():
            os.environ["PROJECT_ROOT"] = str(cand)
            if str(cand / "scripts") not in sys.path:
                sys.path.insert(0, str(cand / "scripts"))
            os.chdir(cand)
            return cand
    raise FileNotFoundError(
        "Could not locate project root with scripts/colab_stream.py. "
        "Rerun §0.3 Pull code from GitHub first."
    )

PROJECT_ROOT = _bootstrap_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)
from colab_stream import run_phase

if os.environ.get("SKIP_PADDLE_TRAIN", "1") != "0":
    print("SKIP_PADDLE_TRAIN=1 — skipping optional PaddleOCR recognition fine-tune")
else:
    print("Phase 04: optional PaddleOCR recognition fine-tune. Logs stream below.")
    sys.stdout.flush()
    env = {
        **os.environ,
        "PYTHON": sys.executable,
        "PYTHONUNBUFFERED": "1",
        "EVAL_USE_GPU": "1",
        "CONFIG_FORCE_GPU": "1",
        "TRAIN_RESUME": "1",
    }
    run_phase("phase_04_train_paddleocr_recognition.sh", env=env, label="Phase 04 PaddleOCR recognition fine-tune")
    print("Phase 04 finished. See results/tables/train_run.json for timing.")


### §4.2 Ablations

Ablation tables from earlier pilot runs have been removed from the active notebook because their runner scripts are not part of the current reproducible pipeline. Add a new ablation script and fresh metrics before restoring this section.


In [ ]:
print("No active ablation phase. Current notebook proceeds to Section D analysis and compile.")


## §5 Analysis & Tables (Section D — stratified, bootstrap, compile)

### §5.1 Bootstrap & Stratified Analysis

Bootstrap CIs, stratified DER, DER-universe ablation, then Table 1. These scripts require fresh JSONL logs from the active model rows.

Skipped when `D_analysis_compile=False`.


In [ ]:
import os, sys
from pathlib import Path

def _bootstrap_project_root():
    import os, sys
    from pathlib import Path
    candidates = []
    if os.environ.get("PROJECT_ROOT"):
        candidates.append(Path(os.environ["PROJECT_ROOT"]))
    if globals().get("REPO_DIR"):
        candidates.append(Path(globals()["REPO_DIR"]))
    candidates.extend([
        Path.cwd(),
        Path("/content/drive/MyDrive/research_ideas_and_how/yoruba_ocr_research"),
        Path("/content/drive/MyDrive/yor_ocr_research"),
        Path("/content/drive/MyDrive/yoruba_ocr_research"),
        Path("/kaggle/working/yoruba_ocr_research"),
    ])
    for cand in candidates:
        if (cand / "scripts" / "colab_stream.py").is_file():
            os.environ["PROJECT_ROOT"] = str(cand)
            if str(cand / "scripts") not in sys.path:
                sys.path.insert(0, str(cand / "scripts"))
            os.chdir(cand)
            return cand
    raise FileNotFoundError(
        "Could not locate project root with scripts/colab_stream.py. "
        "Rerun §0.3 Pull code from GitHub first."
    )

PROJECT_ROOT = _bootstrap_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)
from colab_stream import run_cmd

if os.environ.get("RUN_PLAN_D", "1") != "1":
    print("Section D skipped (D_analysis_compile=False)")
elif os.environ.get("SKIP_ANALYSIS", "0") == "1":
    print("SKIP_ANALYSIS=1")
else:
    for s in ("analyze_stratified_errors.py", "ablate_der_universe.py", "bootstrap_metric_cis.py"):
        run_cmd([sys.executable, f"scripts/{s}"], label=s)


### §5.2 Compile & Audit

Runs only when Section D is enabled.


In [ ]:
import json, os, sys
from pathlib import Path

def _bootstrap_project_root():
    import os, sys
    from pathlib import Path
    candidates = []
    if os.environ.get("PROJECT_ROOT"):
        candidates.append(Path(os.environ["PROJECT_ROOT"]))
    if globals().get("REPO_DIR"):
        candidates.append(Path(globals()["REPO_DIR"]))
    candidates.extend([
        Path.cwd(),
        Path("/content/drive/MyDrive/research_ideas_and_how/yoruba_ocr_research"),
        Path("/content/drive/MyDrive/yor_ocr_research"),
        Path("/content/drive/MyDrive/yoruba_ocr_research"),
        Path("/kaggle/working/yoruba_ocr_research"),
    ])
    for cand in candidates:
        if (cand / "scripts" / "colab_stream.py").is_file():
            os.environ["PROJECT_ROOT"] = str(cand)
            if str(cand / "scripts") not in sys.path:
                sys.path.insert(0, str(cand / "scripts"))
            os.chdir(cand)
            return cand
    raise FileNotFoundError(
        "Could not locate project root with scripts/colab_stream.py. "
        "Rerun §0.3 Pull code from GitHub first."
    )

PROJECT_ROOT = _bootstrap_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)
from colab_stream import run_cmd, run_phase

try:
    from IPython.display import display
except ImportError:
    display = print

if os.environ.get("RUN_PLAN_D", "1") != "1":
    print("Section D compile skipped (D_analysis_compile=False)")
else:
    os.environ["VERIFY_STRICT"] = "1"
    run_phase("phase_09_compile.sh", label="Compile paper tables")

    # Generate research-grade comparison figures from real result artifacts.
    # The script writes 300-DPI PNG plus editable PDF/SVG files.
    run_cmd([sys.executable, "scripts/generate_plots.py"], label="Generate paper figures")

    run_cmd([
        sys.executable, "scripts/diagnose_experiment.py", "checkpoints",
        "--csv", "results/tables/metrics.csv",
        "--report", "results/tables/checkpoint_audit.json",
    ])

    expected = [
        "table1_main_comparison.csv",
        "metrics_summary.csv",
        "bootstrap_metric_cis.csv",
        "bootstrap_pairwise_comparison.csv",
        "stratified_der_by_density.csv",
        "stratified_by_linguistic_features.csv",
        "der_universe_ablation.csv",
        "error_taxonomy.csv",
    ]
    for name in expected:
        p = Path("results/tables") / name
        print(("OK" if p.is_file() else "MISSING"), name)

    # Verify generated paper figures (PNG for notebook display, PDF/SVG for paper editing).
    figure_stems = [
        "model_metrics_comparison",
        "relative_error_reduction",
        "bootstrap_confidence_intervals",
        "stratified_der_by_density",
        "error_taxonomy_distribution",
        "hard_cases_benchmark",
    ]
    for stem in figure_stems:
        for ext in ("png", "pdf", "svg"):
            p = Path("results/tables/figures") / f"{stem}.{ext}"
            print(("OK" if p.is_file() else "MISSING"), f"figure: {stem}.{ext}")

    try:
        df = pd.read_csv("results/tables/metrics_summary.csv")
    except Exception as e:
        raise RuntimeError(
            f"Could not read metrics_summary.csv — re-run Section D: {e}"
        ) from e
    display(df)
    # Display linguistic features stratification
    lf_csv = Path("results/tables/stratified_by_linguistic_features.csv")
    if lf_csv.is_file():
        print("\n=== Stratified by Linguistic Features ===")
        display(pd.read_csv(lf_csv))

    if "phantom" in df.columns and df["phantom"].astype(str).str.lower().isin(["true", "1"]).any():
        print("WARNING: phantom=true rows — do not cite")

    audit = Path("results/tables/checkpoint_audit.json")
    if audit.is_file():
        try:
            a = json.loads(audit.read_text(encoding="utf-8"))
            bad = [r for r in a.get("rows", []) if r.get("status") != "ok"]
            print("checkpoint audit:", "OK" if not bad else f"{len(bad)} issue(s)")
        except json.JSONDecodeError as exc:
            print(f"WARNING: checkpoint_audit.json is malformed or partially written: {exc}")

    rep = Path("results/tables/eval_alignment_report.json")
    if rep.is_file():
        try:
            r = json.loads(rep.read_text(encoding="utf-8"))
            print("eval alignment mismatches:", len(r.get("mismatches", [])))
        except json.JSONDecodeError as exc:
            print(f"WARNING: eval_alignment_report.json is malformed or partially written: {exc}")


### §5.3 Hard-Cases Benchmark

Evaluates model performance on **four linguistically challenging sub-categories** of the Yorùbá test set. These are defined purely from the ground-truth text — no manual annotation required — making them fully reproducible.

| Category | Detection Rule |
|---|---|
| **Named Entities** | Capitalised word mid-sentence OR any capitalised word carrying a diacritic |
| **Numerics** | Line contains at least one digit (currency, dates, percentages) |
| **Historical Orthography** | Contains `sh` or apostrophe-adjoined characters (pre-1974 conventions) |
| **Code-Mixed (Yorùbá–English)** | Line has ≥ 1 diacritic-bearing (Yorùbá) word **and** ≥ 1 pure-ASCII word (len ≥ 3) |

> **Figure 5** shows CER (%) per model across all four categories. Categories with higher CER reveal where each model has structural weaknesses beyond overall performance.

> **Overlap note:** Categories are **intentionally non-exclusive** — a single line may match multiple categories simultaneously (e.g. a line with a digit *and* a capitalised diacritic word matches both *Numerics* and *Named Entities*). Each category is evaluated independently over all matching lines; CER figures are not deduplicated across categories. This is by design — we report per-category difficulty, not a mutually-exclusive partition. See `analyze_stratified_errors.py` → `write_features_csv()` for the aggregation logic.


In [ ]:
from pathlib import Path

try:
    from IPython.display import display, Image as IPImage
except ImportError:
    display = print
    IPImage = None

# ── Hard-Cases Benchmark table ──────────────────────────────────────────
features_csv = Path("results/tables/stratified_by_linguistic_features.csv")
if features_csv.is_file():
    try:
        import pandas as pd
        df_feat = pd.read_csv(features_csv)
        print("=== Hard-Cases Benchmark: CER/WER/DER by Linguistic Feature ===")
        # Pivot to wide form: feature × model for easier comparison
        try:
            pivot = df_feat.pivot_table(
                index="feature",
                columns="model",
                values="cer_pct",
                aggfunc="first",
            )
            display(pivot)
        except Exception as e:
            print(f"Pivot skipped ({e}); showing raw table instead.")
            display(df_feat)
    except Exception as e:
        print(f"Hard-cases table display skipped: {e}")
        print("Figures below can still display if they were generated successfully.")
else:
    print("stratified_by_linguistic_features.csv not yet generated — run Phase 17 first.")

# ── Paper figures for notebook inspection ────────────────────────────────
paper_figures = [
    ("model_metrics_comparison", "Main Metric Performance Comparison"),
    ("relative_error_reduction", "Relative Error Reduction vs English PP-OCR Baseline"),
    ("bootstrap_confidence_intervals", "Bootstrap Metric Confidence Intervals"),
    ("stratified_der_by_density", "Stratified DER by Diacritic Density"),
    ("error_taxonomy_distribution", "Character Diacritic Error Taxonomy"),
    ("hard_cases_benchmark", "Hard-Cases Benchmark — CER by Linguistic Feature Category"),
]
for stem, title in paper_figures:
    fig = Path("results/tables/figures") / f"{stem}.png"
    if fig.is_file():
        print(f"\n{title}")
        if IPImage is not None:
            display(IPImage(filename=str(fig), width=900))
        else:
            print(f"[Figure saved at {fig}]")
    else:
        print(f"{stem}.png not yet generated — run §5.2 Compile & Audit first.")


## §6 Publish

### §6.1 Write research_approach.md


In [ ]:
import sys
from pathlib import Path

def _bootstrap_project_root():
    import os, sys
    from pathlib import Path
    candidates = []
    if os.environ.get("PROJECT_ROOT"):
        candidates.append(Path(os.environ["PROJECT_ROOT"]))
    if globals().get("REPO_DIR"):
        candidates.append(Path(globals()["REPO_DIR"]))
    candidates.extend([
        Path.cwd(),
        Path("/content/drive/MyDrive/research_ideas_and_how/yoruba_ocr_research"),
        Path("/content/drive/MyDrive/yor_ocr_research"),
        Path("/content/drive/MyDrive/yoruba_ocr_research"),
        Path("/kaggle/working/yoruba_ocr_research"),
    ])
    for cand in candidates:
        if (cand / "scripts" / "colab_stream.py").is_file():
            os.environ["PROJECT_ROOT"] = str(cand)
            if str(cand / "scripts") not in sys.path:
                sys.path.insert(0, str(cand / "scripts"))
            os.chdir(cand)
            return cand
    raise FileNotFoundError(
        "Could not locate project root with scripts/colab_stream.py. "
        "Rerun §0.3 Pull code from GitHub first."
    )

PROJECT_ROOT = _bootstrap_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)
from colab_stream import run_cmd

out = Path("research_approach.md")
run_cmd([sys.executable, "scripts/write_research_approach.py", "--output", str(out)])
print("Saved:", out.resolve())
print(out.read_text(encoding="utf-8")[:2000], "...")


### §6.2 Backup results to Drive (Phase 99)


In [ ]:
import os, sys
from pathlib import Path

def _bootstrap_project_root():
    import os, sys
    from pathlib import Path
    candidates = []
    if os.environ.get("PROJECT_ROOT"):
        candidates.append(Path(os.environ["PROJECT_ROOT"]))
    if globals().get("REPO_DIR"):
        candidates.append(Path(globals()["REPO_DIR"]))
    candidates.extend([
        Path.cwd(),
        Path("/content/drive/MyDrive/research_ideas_and_how/yoruba_ocr_research"),
        Path("/content/drive/MyDrive/yor_ocr_research"),
        Path("/content/drive/MyDrive/yoruba_ocr_research"),
        Path("/kaggle/working/yoruba_ocr_research"),
    ])
    for cand in candidates:
        if (cand / "scripts" / "colab_stream.py").is_file():
            os.environ["PROJECT_ROOT"] = str(cand)
            if str(cand / "scripts") not in sys.path:
                sys.path.insert(0, str(cand / "scripts"))
            os.chdir(cand)
            return cand
    raise FileNotFoundError(
        "Could not locate project root with scripts/colab_stream.py. "
        "Rerun §0.3 Pull code from GitHub first."
    )

PROJECT_ROOT = _bootstrap_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)
from colab_stream import run_phase

if not IN_COLAB:
    print("Local — skip or set DRIVE_BACKUP_ROOT")
elif DRIVE_ROOT:
    os.environ["DRIVE_BACKUP_ROOT"] = f"{DRIVE_ROOT}/yoruba_ocr_backups"
    os.environ["BACKUP_EXPERIMENTS"] = "1"
    run_phase("phase_99_backup.sh")
    p = Path("results/tables/.last_drive_backup_path.txt")
    if p.is_file():
        print("Backup at:", p.read_text(encoding="utf-8").strip())
else:
    print("Colab run, but DRIVE_ROOT is not mounted.")


### §6.3 Hugging Face releases

Set `appendix_hf_dataset=True` and/or `appendix_hf_models=True` in Step 1. Requires Colab secret **`HF_TOKEN`** with write access.

- **Dataset** — full benchmark (`data/processed/`) via `publish_hf_dataset.py`
- **Models** — VL-1.6 SFT adapter


In [ ]:
import os, sys
from pathlib import Path

def _bootstrap_project_root():
    import os, sys
    from pathlib import Path
    candidates = []
    if os.environ.get("PROJECT_ROOT"):
        candidates.append(Path(os.environ["PROJECT_ROOT"]))
    if globals().get("REPO_DIR"):
        candidates.append(Path(globals()["REPO_DIR"]))
    candidates.extend([
        Path.cwd(),
        Path("/content/drive/MyDrive/research_ideas_and_how/yoruba_ocr_research"),
        Path("/content/drive/MyDrive/yor_ocr_research"),
        Path("/content/drive/MyDrive/yoruba_ocr_research"),
        Path("/kaggle/working/yoruba_ocr_research"),
    ])
    for cand in candidates:
        if (cand / "scripts" / "colab_stream.py").is_file():
            os.environ["PROJECT_ROOT"] = str(cand)
            if str(cand / "scripts") not in sys.path:
                sys.path.insert(0, str(cand / "scripts"))
            os.chdir(cand)
            return cand
    raise FileNotFoundError(
        "Could not locate project root with scripts/colab_stream.py. "
        "Rerun §0.3 Pull code from GitHub first."
    )

PROJECT_ROOT = _bootstrap_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)
from colab_stream import run_cmd

if os.environ.get("SKIP_HF_DATASET_UPLOAD", "1") == "1":
    print("SKIP_HF_DATASET_UPLOAD=1 — preview card with:")
    print("  python scripts/publish_hf_dataset.py --dry-run")
else:
    repo = Path(REPO_DIR)
    cmd = [sys.executable, "scripts/publish_hf_dataset.py", "--push"]
    repo_id = os.environ.get("HF_DATASET_REPO_ID", "").strip()
    if repo_id:
        cmd.extend(["--repo-id", repo_id])
    if os.environ.get("HF_DATASET_PRIVATE", "0") == "1":
        cmd.append("--private")
    run_cmd(cmd, cwd=repo, label="HF dataset upload")
    manifest = repo / "results/tables/hf_dataset_upload.json"
    if manifest.is_file():
        print(manifest.read_text(encoding="utf-8"))


### §6.4 Upload fine-tuned models

Pushes VL SFT when `appendix_hf_models=True`.


In [ ]:
import os, sys
from pathlib import Path


def _bootstrap_project_root():
    import os, sys
    from pathlib import Path
    candidates = []
    if os.environ.get("PROJECT_ROOT"):
        candidates.append(Path(os.environ["PROJECT_ROOT"]))
    if globals().get("REPO_DIR"):
        candidates.append(Path(globals()["REPO_DIR"]))
    candidates.extend([
        Path.cwd(),
        Path("/content/drive/MyDrive/research_ideas_and_how/yoruba_ocr_research"),
        Path("/content/drive/MyDrive/yor_ocr_research"),
        Path("/content/drive/MyDrive/yoruba_ocr_research"),
        Path("/kaggle/working/yoruba_ocr_research"),
    ])
    for cand in candidates:
        if (cand / "scripts" / "colab_stream.py").is_file():
            os.environ["PROJECT_ROOT"] = str(cand)
            if str(cand / "scripts") not in sys.path:
                sys.path.insert(0, str(cand / "scripts"))
            os.chdir(cand)
            return cand
    raise FileNotFoundError(
        "Could not locate project root with scripts/colab_stream.py. "
        "Rerun §0.3 Pull code from GitHub first."
    )

PROJECT_ROOT = _bootstrap_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)

try:
    from colab_stream import run_cmd
except ImportError:
    from subprocess import run as run_cmd

if os.environ.get("SKIP_HF_MODELS_UPLOAD", "1") == "1":
    print("SKIP_HF_MODELS_UPLOAD=1 — model upload skipped.")
    print("To upload, set appendix_hf_models=True in §0.2 RUN_PLAN and re-run.")
    print("Manual alternative: huggingface-cli upload <repo_id> experiments/paddleocrvl16_sft/")
else:
    repo_id = os.environ.get("HF_MODEL_REPO_ID", "").strip()
    if not repo_id:
        raise ValueError(
            "Set HF_MODEL_REPO_ID env var (e.g. 'sam4rano/paddleocr-vl16-yoruba') before uploading."
        )
    model_dir = Path("experiments/paddleocrvl16_sft")
    if not model_dir.is_dir():
        raise FileNotFoundError(f"Fine-tuned model not found at {model_dir} — run Section B first.")
    cmd = [
        "huggingface-cli", "upload",
        repo_id,
        str(model_dir),
        "--repo-type", "model",
    ]
    if os.environ.get("HF_MODEL_PRIVATE", "0") == "1":
        cmd += ["--private"]
    run_cmd(cmd)
    print(f"Model uploaded to https://huggingface.co/{repo_id}")


### §7.1 Inference demo


In [ ]:
import gc, os
import sys
from pathlib import Path
import torch
from PIL import Image

try:
    from IPython.display import display
except ImportError:
    display = print

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_mem, total_mem = torch.cuda.mem_get_info()
    free_gb = free_mem / (1024 ** 3)
    total_gb = total_mem / (1024 ** 3)
    print(f"CUDA VRAM Info: Free: {free_gb:.2f} GB / Total: {total_gb:.2f} GB")
    if free_gb < 4.0:
        print("WARNING: Low GPU memory. If another model is active, delete it or restart the runtime.")

scripts_path = str(Path.cwd() / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
import paddle_vl_shared

from transformers import AutoProcessor, AutoModel

base_id = "PaddlePaddle/PaddleOCR-VL-1.6"
model_local = Path("experiments/paddleocrvl16_sft")
model_path = str(model_local) if model_local.is_dir() else base_id

test_images = sorted(
    img
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff")
    for img in Path("data/processed/images/test").glob(ext)
)
if not test_images:
    print("No test images found — run Section 1 first.")
else:
    test_image = test_images[0]
    print(f"Sample image: {test_image.name}")
    print(f"Loading model from: {model_path}")

    # Note: trust_remote_code=True is required because custom vision-language models
    # define their specific architectures/layers in Python code hosted on HF Hub.
    processor = AutoProcessor.from_pretrained(base_id, trust_remote_code=True)
    model = AutoModel.from_pretrained(
        model_path,
        trust_remote_code=True,
        dtype=paddle_vl_shared.select_torch_dtype()[0],
        device_map="auto" if torch.cuda.is_available() else None,
    ).eval()

    image = Image.open(test_image).convert("RGB")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": paddle_vl_shared.USER_TEXT_OCR_YORUBA},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    device = next(model.parameters()).device
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(device)

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=128)
        trimmed = [ids[inputs.input_ids.shape[1]:] for ids in out_ids]
        prediction = processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

    print(f"Model   : {base_id} (zero-shot/fine-tuned)")
    print(f"Prediction: {prediction}")
    display(image)


In [ ]:
# Final Sanity Check: Table 1 Validation
import pandas as pd
from pathlib import Path

try:
    from IPython.display import display
except ImportError:
    display = print

summary_path = Path("results/tables/metrics_summary.csv")
if not summary_path.is_file():
    print("❌ metrics_summary.csv not found! Run Section D (compile) first.")
else:
    try:
        df = pd.read_csv(summary_path)
    except Exception as e:
        raise RuntimeError(
            f"Could not read metrics_summary.csv — re-run Section D: {e}"
        ) from e
    print("=== Table 1 — Compiled Metrics Summary ===")
    print(df.to_string(index=False))
    print("\n=== Expected Model Row Checklist ===")

    # Active model rows in the paper
    expected_models = [
        ("paddleocr_en_pretrained", "PaddleOCR EN pretrained"),
        ("paddleocrvl16_zero_shot",   "PaddleOCR-VL-1.6 (zero-shot)"),
        ("glm_ocr_zero_shot",          "GLM-OCR (zero-shot)"),
        ("paddleocrvl16_sft", "PaddleOCR-VL-1.6 (SFT, optional)"),
    ]

    missing = []
    for model_key, display_name in expected_models:
        present = model_key in df["model_label"].values
        status = "✅ PASS" if present else "⏳ PENDING"
        if not present:
            missing.append(model_key)
        print(f"{status:<10} | {display_name}")

    print()
    if not missing:
        print("✅ All active model rows present — Table 1 is complete.")
    else:
        print("⏳ Clean active results are missing model rows that still need full evaluation:")
        for model_key in missing:
            print(f"  - {model_key}")
        print("Re-run Section A evaluation(s) on GPU, then Section D compile/alignment.")
